In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import roc_curve, auc, roc_auc_score

from sklearn.model_selection import GridSearchCV

# Grafici
import matplotlib.pyplot as plt
import seaborn as sns

# Keras
from scikeras.wrappers import KerasClassifier
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.regularizers import l2, l1
from keras.layers import Dropout
from sklearn.model_selection import RandomizedSearchCV
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.models import load_model

In [15]:
####
# DATASET
####
df = pd.read_parquet("df_pulito.parquet")
# df["bmi"] = (df["Physical-Weight"] / (df["Physical-Height"] ** 2)) * 703

# Train Test sii
pciat_cols = [f"PCIAT-PCIAT_{i:02d}" for i in range(1, 21)]
X = df.drop(columns=["sii", "risultati",
            "PCIAT-PCIAT_Total", "id"] + pciat_cols)
y = df["sii"]  # per valutare sii, praticamente classificazione
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=222)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Test Ore
X_ore = df.drop(columns=["risultati", "id",
                "PreInt_EduHx-computerinternet_hoursday"])
y_ore = df['PreInt_EduHx-computerinternet_hoursday']

X_train_ore, X_test_ore, y_train_ore, y_test_ore = train_test_split(
    X_ore, y_ore, test_size=0.3, random_state=222)

X_train_ore_scaled = scaler.fit_transform(X_train_ore)
X_test_ore_scaled = scaler.transform(X_test_ore)


# Train Test Attività Fisica
X_af = df.drop(columns=["risultati", "BIA-BIA_Activity_Level_num", "id"])
y_af = df['BIA-BIA_Activity_Level_num'] - 1

X_train_af, X_test_af, y_train_af, y_test_af = train_test_split(
    X_af, y_af, test_size=0.3, random_state=222)

X_train_af_scaled = scaler.fit_transform(X_train_af)
X_test_af_scaled = scaler.transform(X_test_af)

# VALUTAZIONE sii
---

In [7]:
######
# MODELLO SENZA REGOLARIZZAZIONE
######

####
# DEFINIZIONE & TRAINING RETE NEURALE
####


###
# Builder del modello
###

def Costruttore_NOREG(hidden_layer_sizes, classes, features, activation='relu', learning_rate=0.001):
    '''
    hidden_layer_sizes accetta un range di valori, e con ognuno ci fa un layer
    '''
    # Sequential = tipo del modello. Connette strati di neuroni in maniera sequenziale (uno dopo l'altro)
    model = Sequential()

    # Strato di input
    model.add(keras.layers.Input(shape=(features,)))

    # Hidden layers:
    # Dense:: indica che lo strato è Fully Connected
    # units: sono il numero di neuroni nello strato
    # activation:: selezione della funzione di attivazione (RELU, ELU etc)
    # relu = Rectified Linear Unit -> f(x) = max(0, x); serve per apprendere pattern non lineari complessi
    for units in hidden_layer_sizes:
        model.add(Dense(units, activation=activation))

    # Layer di Output, a 4 neuroni (numero categorie)
    # softmax converte i valori grezzi finali in probabilità, così che per ogni osservazione la somma dei suoi punteggi di classe è 1
    model.add(Dense(classes, activation='softmax'))

    # Si usa Adam per ottimizzare, con learning rate che può essere scelto a mano
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(loss='sparse_categorical_crossentropy',  # funzione di perdita per tasks categorici in cui le categorie sono numeri (non binari)
                  optimizer=optimizer,  # adam = Adaptive Moment Estimation, discesa stocastoca del gradiente
                  metrics=['accuracy'])
    return model


# Definizione Early Stopping; il checklpoint viene fatto manualmente dopo la grid search
es = EarlyStopping(monitor='val_loss',  # misura che viene valutata per early stopping
                   min_delta=0,  # minimo cambiamento nella misura monitorata per essere considerato miglioramento
                   patience=20,  # dopo tot epoche senza cambiamenti il processo si ferma
                   verbose=0,
                   restore_best_weights=True,
                   start_from_epoch=10  # Ho deciso di dare al modello 10 epoche "di riscaldamento"; dopo 10 epoche inizia a controllare early stopping
                   )

###
# Ricerca parametri migliori
###

Classificatore = KerasClassifier(
    model=Costruttore_NOREG,  # la size dell'hidden layer è un placeholder
    model__classes=4,
    model__features=53,
    random_state=222,
    verbose=0)


# Spazio Iperparametri
parametri = {
    # Profondità: shallow, medium, deep
    'model__hidden_layer_sizes': [(64,), (128, 64), (64, 32), (128, 64, 32)],
    # Learnign rates
    'model__learning_rate': [0.1, 0.01, 0.001, 0.0001]
}

# Gridsearch
rs_NOREG = RandomizedSearchCV(
    estimator=Classificatore,
    param_distributions=parametri,
    n_iter=15,
    scoring='f1_macro',
    cv=3,
    random_state=222,
    verbose=2,
    n_jobs=-1
)

rs_NOREG.fit(X_train_scaled, y_train, epochs=500,
             validation_split=0.1, callbacks=[es])  # Ricerca effettiva

best_model_NOREG = rs_NOREG.best_estimator_  # estrazione modello migliore
# salvataggio modello migliore
best_model_NOREG.model_.save('best_model_NOREG.keras')

# Valutazione modello


# Parametri e stats modello migliore
for parametro, valore in rs_NOREG.best_params_.items():
    print(f" - {parametro}: {valore}")
best_model_NOREG.model_.summary()
y_pred = best_model_NOREG.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 15 candidates, totalling 45 fits
 - model__learning_rate: 0.01
 - model__hidden_layer_sizes: (128, 64, 32)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         6,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,142 (203.68 KB)

 Trainable params: 17,380 (67.89 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 34,762 (135.79 KB)

              precision    recall  f1-score   support

         0.0       0.81      0.75      0.78      1742
         1.0       0.29      0.40      0.33       450
         2.0       0.24      0.21      0.23       261
         3.0       0.12      0.06      0.08        16

    accuracy                           0.62      2469
   macro avg       0.37      0.36      0.36      2469
weighted avg       0.65      0.62      0.63      2469



In [8]:
######
# MODELLO L1 LASSO
######

####
# DEFINIZIONE & TRAINING RETE NEURALE
####

###
# Builder del modello
###

def Costruttore_L1(hidden_layer_sizes, classes, features, activation='relu', learning_rate=0.001, l1_rate=0.001):
    '''
    hidden_layer_sizes accetta un range di valori, e con ognuno ci fa un layer.
    l1_rate determina l'intensità della penalizzazione sui pesi.
    '''
    model = Sequential()

    # Strato di input
    model.add(keras.layers.Input(shape=(features,)))

    # Hidden layers
    for units in hidden_layer_sizes:
        # Aggiunta del parametro kernel_regularizer
        model.add(Dense(units, activation=activation,
                  kernel_regularizer=l1(l1_rate)))

    # Layer di Output
    model.add(Dense(classes, activation='softmax'))

    # Ottimizzatore
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])
    return model


# Definizione Early Stopping
es = EarlyStopping(monitor='val_loss',
                   min_delta=0,
                   patience=20,
                   verbose=0,
                   restore_best_weights=True,
                   start_from_epoch=10
                   )

###
# Ricerca parametri migliori
###

Classificatore = KerasClassifier(
    model=Costruttore_L1,
    model__classes=4,
    model__features=53,
    random_state=222,
    verbose=0)


# Spazio Iperparametri
parametri = {
    # Profondità: shallow, medium, deep
    'model__hidden_layer_sizes': [(64,), (128, 64), (64, 32), (128, 64, 32)],
    # Learning rates
    'model__learning_rate': [0.1, 0.01, 0.001, 0.0001],
    # Tassi di regolarizzazione L1
    'model__l1_rate': [0.01, 0.001, 0.0001]
}

# Gridsearch
rs_L1 = RandomizedSearchCV(
    estimator=Classificatore,
    param_distributions=parametri,
    n_iter=15,
    scoring='f1_macro',
    cv=3,
    random_state=222,
    verbose=2,
    n_jobs=-1
)

rs_L1.fit(X_train_scaled, y_train, epochs=500,
          validation_split=0.1, callbacks=[es])

best_model_L1 = rs_L1.best_estimator_

# salvataggio modello migliore
best_model_L1.model_.save('best_model_L1.keras')

# Parametri e stats modello migliore
for parametro, valore in rs_L1.best_params_.items():
    print(f" - {parametro}: {valore}")
best_model_L1.model_.summary()
y_pred = best_model_L1.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 15 candidates, totalling 45 fits
 - model__learning_rate: 0.0001
 - model__l1_rate: 0.001
 - model__hidden_layer_sizes: (64, 32)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │         3,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,006 (66.43 KB)

 Trainable params: 5,668 (22.14 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 11,338 (44.29 KB)

              precision    recall  f1-score   support

         0.0       0.80      0.87      0.83      1742
         1.0       0.33      0.31      0.32       450
         2.0       0.36      0.23      0.28       261
         3.0       0.00      0.00      0.00        16

    accuracy                           0.69      2469
   macro avg       0.37      0.35      0.36      2469
weighted avg       0.66      0.69      0.68      2469



c:\Users\Riccardo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Riccardo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Riccardo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [9]:
######
# MODELLO L2 RIDGE
######

####
# DEFINIZIONE & TRAINING RETE NEURALE
####

###
# Builder del modello
###

def Costruttore_L2(hidden_layer_sizes, classes, features, activation='relu', learning_rate=0.001, l2_rate=0.001):
    '''
    hidden_layer_sizes accetta un range di valori, e con ognuno ci fa un layer.
    l2_rate determina l'intensità della penalizzazione sui pesi (Ridge).
    '''
    model = Sequential()

    # Strato di input
    model.add(keras.layers.Input(shape=(features,)))

    # Hidden layers
    for units in hidden_layer_sizes:
        # Aggiunta del parametro kernel_regularizer con funzione l2
        model.add(Dense(units, activation=activation,
                  kernel_regularizer=l2(l2_rate)))

    # Layer di Output
    model.add(Dense(classes, activation='softmax'))

    # Ottimizzatore
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])
    return model


# Definizione Early Stopping
es = EarlyStopping(monitor='val_loss',
                   min_delta=0,
                   patience=20,
                   verbose=0,
                   restore_best_weights=True,
                   start_from_epoch=10
                   )

###
# Ricerca parametri migliori
###

Classificatore = KerasClassifier(
    model=Costruttore_L2,
    model__classes=4,
    model__features=53,
    random_state=222,
    verbose=0)


# Spazio Iperparametri
parametri = {
    # Profondità: shallow, medium, deep
    'model__hidden_layer_sizes': [(64,), (128, 64), (64, 32), (128, 64, 32)],
    # Learning rates
    'model__learning_rate': [0.1, 0.01, 0.001, 0.0001],
    # Tassi di regolarizzazione L2
    'model__l2_rate': [0.01, 0.001, 0.0001]
}


# Gridsearch
rs_L2 = RandomizedSearchCV(
    estimator=Classificatore,
    param_distributions=parametri,
    n_iter=15,
    scoring='f1_macro',
    cv=3,
    random_state=222,
    verbose=2,
    n_jobs=-1
)

rs_L2.fit(X_train_scaled, y_train, epochs=500,
          validation_split=0.1, callbacks=[es])

best_model_L2 = rs_L2.best_estimator_

# salvataggio modello migliore
best_model_L2.model_.save('best_model_L2.keras')

# Parametri e stats modello migliore
for parametro, valore in rs_L2.best_params_.items():
    print(f" - {parametro}: {valore}")
best_model_L2.model_.summary()
y_pred = best_model_L2.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 15 candidates, totalling 45 fits
 - model__learning_rate: 0.0001
 - model__l2_rate: 0.01
 - model__hidden_layer_sizes: (64, 32)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                 │ (None, 64)             │         3,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,006 (66.43 KB)

 Trainable params: 5,668 (22.14 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 11,338 (44.29 KB)

              precision    recall  f1-score   support

         0.0       0.79      0.86      0.82      1742
         1.0       0.34      0.28      0.31       450
         2.0       0.32      0.23      0.26       261
         3.0       0.00      0.00      0.00        16

    accuracy                           0.68      2469
   macro avg       0.36      0.34      0.35      2469
weighted avg       0.65      0.68      0.67      2469



c:\Users\Riccardo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Riccardo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Riccardo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
######
# MODELLO DROPOUT
######

####
# DEFINIZIONE & TRAINING RETE NEURALE
####

###
# Builder del modello
###

def Costruttore_DROPOUT(hidden_layer_sizes, classes, features, activation='relu', learning_rate=0.001, dropout_rate=0.2):
    '''
    hidden_layer_sizes accetta un range di valori, e con ognuno ci fa un layer.
    dropout_rate determina la percentuale di neuroni da disattivare casualmente ad ogni iterazione.
    '''
    model = Sequential()

    # Strato di input
    model.add(keras.layers.Input(shape=(features,)))

    # Hidden layers
    for units in hidden_layer_sizes:
        # Aggiunta dello strato denso standard
        model.add(Dense(units, activation=activation))
        # Aggiunta dello strato di Dropout per la regolarizzazione
        model.add(Dropout(dropout_rate))

    # Layer di Output
    model.add(Dense(classes, activation='softmax'))

    # Ottimizzatore
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])
    return model


# Definizione Early Stopping
es = EarlyStopping(monitor='val_loss',
                   min_delta=0,
                   patience=20,
                   verbose=0,
                   restore_best_weights=True,
                   start_from_epoch=10
                   )

###
# Ricerca parametri migliori
###

Classificatore = KerasClassifier(
    model=Costruttore_DROPOUT,
    model__classes=4,
    model__features=53,
    random_state=222,
    verbose=0)


# Spazio Iperparametri
parametri = {
    # Profondità: shallow, medium, deep
    'model__hidden_layer_sizes': [(64,), (128, 64), (64, 32), (128, 64, 32)],
    # Learning rates
    'model__learning_rate': [0.1, 0.01, 0.001, 0.0001],
    # Tassi di Dropout (dal 10% al 50%)
    'model__dropout_rate': [0.1, 0.2, 0.3, 0.4, 0.5]
}

# Gridsearch
rs_DROPOUT = RandomizedSearchCV(
    estimator=Classificatore,
    param_distributions=parametri,
    n_iter=15,
    scoring='f1_macro',
    cv=3,
    random_state=222,
    verbose=2,
    n_jobs=-1
)

rs_DROPOUT.fit(X_train_scaled, y_train, epochs=500,
               validation_split=0.1, callbacks=[es])

best_model_DROPOUT = rs_DROPOUT.best_estimator_

# salvataggio modello migliore
best_model_DROPOUT.model_.save('best_model_DROPOUT.keras')

# Parametri e stats modello migliore
for parametro, valore in rs_DROPOUT.best_params_.items():
    print(f" - {parametro}: {valore}")
best_model_DROPOUT.model_.summary()
y_pred = best_model_DROPOUT.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 15 candidates, totalling 45 fits
 - model__learning_rate: 0.001
 - model__hidden_layer_sizes: (64, 32)
 - model__dropout_rate: 0.3


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 64)             │         3,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,006 (66.43 KB)

 Trainable params: 5,668 (22.14 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 11,338 (44.29 KB)

              precision    recall  f1-score   support

         0.0       0.80      0.83      0.81      1742
         1.0       0.33      0.37      0.35       450
         2.0       0.32      0.19      0.24       261
         3.0       0.00      0.00      0.00        16

    accuracy                           0.67      2469
   macro avg       0.36      0.35      0.35      2469
weighted avg       0.66      0.67      0.66      2469



In [11]:
####
# COMPARISON MODELLI
####

# Caricamento modelli migliori
best_NOREG_model = load_model('best_model_NOREG.keras')
best_L1_model = load_model('best_model_L1.keras')
best_L2_model = load_model('best_model_L2.keras')
best_DROPOUT_model = load_model('best_model_DROPOUT.keras')

# Valutazioni di Keras
loss_NOREG, acc_NOREG = best_NOREG_model.evaluate(
    X_test_scaled, y_test, verbose=0)
loss_L1, acc_L1 = best_L1_model.evaluate(X_test_scaled, y_test, verbose=0)
loss_L2, acc_L2 = best_L2_model.evaluate(X_test_scaled, y_test, verbose=0)
loss_DROPOUT, acc_DROPOUT = best_DROPOUT_model.evaluate(
    X_test_scaled, y_test, verbose=0)

# Predizioni
pred_NOREG = np.argmax(best_NOREG_model.predict(
    X_test_scaled, verbose=0), axis=1)
pred_L1 = np.argmax(best_L1_model.predict(X_test_scaled, verbose=0), axis=1)
pred_L2 = np.argmax(best_L2_model.predict(X_test_scaled, verbose=0), axis=1)
pred_DROPOUT = np.argmax(best_DROPOUT_model.predict(
    X_test_scaled, verbose=0), axis=1)

# Calcolo F1-Score Macro
f1_NOREG = f1_score(y_test, pred_NOREG, average='macro')
f1_L1 = f1_score(y_test, pred_L1, average='macro')
f1_L2 = f1_score(y_test, pred_L2, average='macro')
f1_DROPOUT = f1_score(y_test, pred_DROPOUT, average='macro')

# Stampa finale
print("### COMPARAZIONE FINALE MODELLI DNN ###\n")
print(f"{'Modello':<25} | {'Loss':<8} | {'Accuracy':<10} | {'F1-Macro':<10}")
print("-" * 65)
print(f"{'No Regolarizzazione':<25} | {loss_NOREG:<8.4f} | {acc_NOREG:<10.4f} | {f1_NOREG:<10.4f}")
print(f"{'L1 (Lasso)':<25} | {loss_L1:<8.4f} | {acc_L1:<10.4f} | {f1_L1:<10.4f}")
print(f"{'L2 (Ridge)':<25} | {loss_L2:<8.4f} | {acc_L2:<10.4f} | {f1_L2:<10.4f}")
print(f"{'Dropout':<25} | {loss_DROPOUT:<8.4f} | {acc_DROPOUT:<10.4f} | {f1_DROPOUT:<10.4f}")

### COMPARAZIONE FINALE MODELLI DNN ###

Modello                   | Loss     | Accuracy   | F1-Macro  
-----------------------------------------------------------------
No Regolarizzazione       | 7.5266   | 0.6229     | 0.3554    
L1 (Lasso)                | 0.9398   | 0.6926     | 0.3579    
L2 (Ridge)                | 0.9980   | 0.6837     | 0.3491    
Dropout                   | 1.1618   | 0.6715     | 0.3501    


# VALUTAZIONE ORE
---

In [12]:
from scikeras.wrappers import KerasClassifier
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.regularizers import l2, l1
from sklearn.model_selection import RandomizedSearchCV
from keras.callbacks import EarlyStopping
from keras.models import load_model
from sklearn.metrics import classification_report, f1_score
import numpy as np

# Definizione Early Stopping per i modelli "ore"
es_ore = EarlyStopping(
    monitor='val_loss',
    min_delta=0,
    patience=20,
    verbose=0,
    restore_best_weights=True,
    start_from_epoch=10
)

# Spazio Iperparametri condiviso (Ore)
parametri_ore = {
    'model__hidden_layer_sizes': [(64,), (128, 64), (64, 32), (128, 64, 32)],
    'model__learning_rate': [0.1, 0.01, 0.001, 0.0001]
}

# 1. MODELLO SENZA REGOLARIZZAZIONE (ORE)


def Costruttore_NOREG_ore(hidden_layer_sizes, classes, features, activation='relu', learning_rate=0.001):
    model = Sequential()
    model.add(keras.layers.Input(shape=(features,)))
    for units in hidden_layer_sizes:
        model.add(Dense(units, activation=activation))
    model.add(Dense(classes, activation='softmax'))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer, metrics=['accuracy'])
    return model


Classificatore_NOREG_ore = KerasClassifier(
    model=Costruttore_NOREG_ore,
    model__classes=4,
    model__features=74,
    random_state=222,
    verbose=0
)

rs_NOREG_ore = RandomizedSearchCV(
    estimator=Classificatore_NOREG_ore,
    param_distributions=parametri_ore,
    n_iter=20,
    scoring='f1_macro',
    cv=3,
    random_state=222,
    verbose=2,
    n_jobs=-1
)

rs_NOREG_ore.fit(X_train_ore_scaled, y_train_ore, epochs=200,
                 validation_split=0.1, callbacks=[es_ore])
best_model_NOREG_ore = rs_NOREG_ore.best_estimator_
best_model_NOREG_ore.model_.save('best_model_NOREG_ore.keras')


# 2. MODELLO L1 LASSO (ORE)
def Costruttore_L1_ore(hidden_layer_sizes, classes, features, activation='relu', learning_rate=0.001, l1_rate=0.001):
    model = Sequential()
    model.add(keras.layers.Input(shape=(features,)))
    for units in hidden_layer_sizes:
        model.add(Dense(units, activation=activation,
                  kernel_regularizer=l1(l1_rate)))
    model.add(Dense(classes, activation='softmax'))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer, metrics=['accuracy'])
    return model


parametri_L1_ore = parametri_ore.copy()
parametri_L1_ore['model__l1_rate'] = [0.01, 0.001, 0.0001]

Classificatore_L1_ore = KerasClassifier(
    model=Costruttore_L1_ore,
    model__classes=4,
    model__features=74,
    random_state=222,
    verbose=0
)

rs_L1_ore = RandomizedSearchCV(
    estimator=Classificatore_L1_ore,
    param_distributions=parametri_L1_ore,
    n_iter=20,
    scoring='f1_macro',
    cv=3,
    random_state=222,
    verbose=2,
    n_jobs=-1
)

rs_L1_ore.fit(X_train_ore_scaled, y_train_ore, epochs=200,
              validation_split=0.1, callbacks=[es_ore])
best_model_L1_ore = rs_L1_ore.best_estimator_
best_model_L1_ore.model_.save('best_model_L1_ore.keras')


# 3. MODELLO L2 RIDGE (ORE)
def Costruttore_L2_ore(hidden_layer_sizes, classes, features, activation='relu', learning_rate=0.001, l2_rate=0.001):
    model = Sequential()
    model.add(keras.layers.Input(shape=(features,)))
    for units in hidden_layer_sizes:
        model.add(Dense(units, activation=activation,
                  kernel_regularizer=l2(l2_rate)))
    model.add(Dense(classes, activation='softmax'))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer, metrics=['accuracy'])
    return model


parametri_L2_ore = parametri_ore.copy()
parametri_L2_ore['model__l2_rate'] = [0.01, 0.001, 0.0001]

Classificatore_L2_ore = KerasClassifier(
    model=Costruttore_L2_ore,
    model__classes=4,
    model__features=74,
    random_state=222,
    verbose=0
)

rs_L2_ore = RandomizedSearchCV(
    estimator=Classificatore_L2_ore,
    param_distributions=parametri_L2_ore,
    n_iter=20,
    scoring='f1_macro',
    cv=3,
    random_state=222,
    verbose=2,
    n_jobs=-1
)

rs_L2_ore.fit(X_train_ore_scaled, y_train_ore, epochs=200,
              validation_split=0.1, callbacks=[es_ore])
best_model_L2_ore = rs_L2_ore.best_estimator_
best_model_L2_ore.model_.save('best_model_L2_ore.keras')


# 4. MODELLO DROPOUT (ORE)
def Costruttore_DROPOUT_ore(hidden_layer_sizes, classes, features, activation='relu', learning_rate=0.001, dropout_rate=0.2):
    model = Sequential()
    model.add(keras.layers.Input(shape=(features,)))
    for units in hidden_layer_sizes:
        model.add(Dense(units, activation=activation))
        model.add(Dropout(dropout_rate))
    model.add(Dense(classes, activation='softmax'))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer, metrics=['accuracy'])
    return model


parametri_DROPOUT_ore = parametri_ore.copy()
parametri_DROPOUT_ore['model__dropout_rate'] = [0.1, 0.2, 0.3, 0.4, 0.5]

Classificatore_DROPOUT_ore = KerasClassifier(
    model=Costruttore_DROPOUT_ore,
    model__classes=4,
    model__features=74,
    random_state=222,
    verbose=0
)

rs_DROPOUT_ore = RandomizedSearchCV(
    estimator=Classificatore_DROPOUT_ore,
    param_distributions=parametri_DROPOUT_ore,
    n_iter=20,
    scoring='f1_macro',
    cv=3,
    random_state=222,
    verbose=2,
    n_jobs=-1
)

rs_DROPOUT_ore.fit(X_train_ore_scaled, y_train_ore, epochs=200,
                   validation_split=0.1, callbacks=[es_ore])
best_model_DROPOUT_ore = rs_DROPOUT_ore.best_estimator_
best_model_DROPOUT_ore.model_.save('best_model_DROPOUT_ore.keras')


# COMPARAZIONE FINALE MODELLI "ORE"
best_NOREG_ore_loaded = load_model('best_model_NOREG_ore.keras')
best_L1_ore_loaded = load_model('best_model_L1_ore.keras')
best_L2_ore_loaded = load_model('best_model_L2_ore.keras')
best_DROPOUT_ore_loaded = load_model('best_model_DROPOUT_ore.keras')

loss_NOREG_ore, acc_NOREG_ore = best_NOREG_ore_loaded.evaluate(
    X_test_ore_scaled, y_test_ore, verbose=0)
loss_L1_ore, acc_L1_ore = best_L1_ore_loaded.evaluate(
    X_test_ore_scaled, y_test_ore, verbose=0)
loss_L2_ore, acc_L2_ore = best_L2_ore_loaded.evaluate(
    X_test_ore_scaled, y_test_ore, verbose=0)
loss_DROPOUT_ore, acc_DROPOUT_ore = best_DROPOUT_ore_loaded.evaluate(
    X_test_ore_scaled, y_test_ore, verbose=0)

pred_NOREG_ore = np.argmax(best_NOREG_ore_loaded.predict(
    X_test_ore_scaled, verbose=0), axis=1)
pred_L1_ore = np.argmax(best_L1_ore_loaded.predict(
    X_test_ore_scaled, verbose=0), axis=1)
pred_L2_ore = np.argmax(best_L2_ore_loaded.predict(
    X_test_ore_scaled, verbose=0), axis=1)
pred_DROPOUT_ore = np.argmax(best_DROPOUT_ore_loaded.predict(
    X_test_ore_scaled, verbose=0), axis=1)

f1_NOREG_ore = f1_score(y_test_ore, pred_NOREG_ore, average='macro')
f1_L1_ore = f1_score(y_test_ore, pred_L1_ore, average='macro')
f1_L2_ore = f1_score(y_test_ore, pred_L2_ore, average='macro')
f1_DROPOUT_ore = f1_score(y_test_ore, pred_DROPOUT_ore, average='macro')

print("\n### COMPARAZIONE FINALE MODELLI DNN (ORE) ###\n")
print(f"{'Modello':<25} | {'Loss':<8} | {'Accuracy':<10} | {'F1-Macro':<10}")
print("-" * 65)
print(f"{'No Regolarizzazione':<25} | {loss_NOREG_ore:<8.4f} | {acc_NOREG_ore:<10.4f} | {f1_NOREG_ore:<10.4f}")
print(f"{'L1 (Lasso)':<25} | {loss_L1_ore:<8.4f} | {acc_L1_ore:<10.4f} | {f1_L1_ore:<10.4f}")
print(f"{'L2 (Ridge)':<25} | {loss_L2_ore:<8.4f} | {acc_L2_ore:<10.4f} | {f1_L2_ore:<10.4f}")
print(f"{'Dropout':<25} | {loss_DROPOUT_ore:<8.4f} | {acc_DROPOUT_ore:<10.4f} | {f1_DROPOUT_ore:<10.4f}")

Fitting 3 folds for each of 16 candidates, totalling 48 fits


c:\Users\Riccardo\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 16 is smaller than n_iter=20. Running 16 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits


c:\Users\Riccardo\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



### COMPARAZIONE FINALE MODELLI DNN (ORE) ###

Modello                   | Loss     | Accuracy   | F1-Macro  
-----------------------------------------------------------------
No Regolarizzazione       | 1.3694   | 0.5079     | 0.4272    
L1 (Lasso)                | 1.3454   | 0.5318     | 0.4307    
L2 (Ridge)                | 1.3813   | 0.5136     | 0.4285    
Dropout                   | 1.3643   | 0.5144     | 0.4253    


# VALUTAZIONE ATTIVITA FISICA
---

In [16]:
# Definizione Early Stopping per i modelli "af"
es_af = EarlyStopping(
    monitor='val_loss',
    min_delta=0,
    patience=20,
    verbose=0,
    restore_best_weights=True,
    start_from_epoch=10
)

# Spazio Iperparametri condiviso (AF)
parametri_af = {
    'model__hidden_layer_sizes': [(64,), (128, 64), (64, 32), (128, 64, 32)],
    'model__learning_rate': [0.1, 0.01, 0.001, 0.0001]
}

# 1. MODELLO SENZA REGOLARIZZAZIONE (AF)


def Costruttore_NOREG_af(hidden_layer_sizes, classes, features, activation='relu', learning_rate=0.001):
    model = Sequential()
    model.add(keras.layers.Input(shape=(features,)))
    for units in hidden_layer_sizes:
        model.add(Dense(units, activation=activation))
    model.add(Dense(classes, activation='softmax'))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer, metrics=['accuracy'])
    return model


Classificatore_NOREG_af = KerasClassifier(
    model=Costruttore_NOREG_af,
    model__classes=5,
    model__features=74,
    random_state=222,
    verbose=0
)

rs_NOREG_af = RandomizedSearchCV(
    estimator=Classificatore_NOREG_af,
    param_distributions=parametri_af,
    n_iter=20,
    scoring='f1_macro',
    cv=3,
    random_state=222,
    verbose=2,
    n_jobs=-1
)

rs_NOREG_af.fit(X_train_af_scaled, y_train_af, epochs=200,
                validation_split=0.1, callbacks=[es_af])
best_model_NOREG_af = rs_NOREG_af.best_estimator_
best_model_NOREG_af.model_.save('best_model_NOREG_af.keras')


# 2. MODELLO L1 LASSO (AF)
def Costruttore_L1_af(hidden_layer_sizes, classes, features, activation='relu', learning_rate=0.001, l1_rate=0.001):
    model = Sequential()
    model.add(keras.layers.Input(shape=(features,)))
    for units in hidden_layer_sizes:
        model.add(Dense(units, activation=activation,
                  kernel_regularizer=l1(l1_rate)))
    model.add(Dense(classes, activation='softmax'))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer, metrics=['accuracy'])
    return model


parametri_L1_af = parametri_af.copy()
parametri_L1_af['model__l1_rate'] = [0.01, 0.001, 0.0001]

Classificatore_L1_af = KerasClassifier(
    model=Costruttore_L1_af,
    model__classes=5,
    model__features=74,
    random_state=222,
    verbose=0
)

rs_L1_af = RandomizedSearchCV(
    estimator=Classificatore_L1_af,
    param_distributions=parametri_L1_af,
    n_iter=20,
    scoring='f1_macro',
    cv=3,
    random_state=222,
    verbose=2,
    n_jobs=-1
)

rs_L1_af.fit(X_train_af_scaled, y_train_af, epochs=200,
             validation_split=0.1, callbacks=[es_af])
best_model_L1_af = rs_L1_af.best_estimator_
best_model_L1_af.model_.save('best_model_L1_af.keras')


# 3. MODELLO L2 RIDGE (AF)
def Costruttore_L2_af(hidden_layer_sizes, classes, features, activation='relu', learning_rate=0.001, l2_rate=0.001):
    model = Sequential()
    model.add(keras.layers.Input(shape=(features,)))
    for units in hidden_layer_sizes:
        model.add(Dense(units, activation=activation,
                  kernel_regularizer=l2(l2_rate)))
    model.add(Dense(classes, activation='softmax'))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer, metrics=['accuracy'])
    return model


parametri_L2_af = parametri_af.copy()
parametri_L2_af['model__l2_rate'] = [0.01, 0.001, 0.0001]

Classificatore_L2_af = KerasClassifier(
    model=Costruttore_L2_af,
    model__classes=5,
    model__features=74,
    random_state=222,
    verbose=0
)

rs_L2_af = RandomizedSearchCV(
    estimator=Classificatore_L2_af,
    param_distributions=parametri_L2_af,
    n_iter=20,
    scoring='f1_macro',
    cv=3,
    random_state=222,
    verbose=2,
    n_jobs=-1
)

rs_L2_af.fit(X_train_af_scaled, y_train_af, epochs=200,
             validation_split=0.1, callbacks=[es_af])
best_model_L2_af = rs_L2_af.best_estimator_
best_model_L2_af.model_.save('best_model_L2_af.keras')


# 4. MODELLO DROPOUT (AF)
def Costruttore_DROPOUT_af(hidden_layer_sizes, classes, features, activation='relu', learning_rate=0.001, dropout_rate=0.2):
    model = Sequential()
    model.add(keras.layers.Input(shape=(features,)))
    for units in hidden_layer_sizes:
        model.add(Dense(units, activation=activation))
        model.add(Dropout(dropout_rate))
    model.add(Dense(classes, activation='softmax'))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer, metrics=['accuracy'])
    return model


parametri_DROPOUT_af = parametri_af.copy()
parametri_DROPOUT_af['model__dropout_rate'] = [0.1, 0.2, 0.3, 0.4, 0.5]

Classificatore_DROPOUT_af = KerasClassifier(
    model=Costruttore_DROPOUT_af,
    model__classes=5,
    model__features=74,
    random_state=222,
    verbose=0
)

rs_DROPOUT_af = RandomizedSearchCV(
    estimator=Classificatore_DROPOUT_af,
    param_distributions=parametri_DROPOUT_af,
    n_iter=20,
    scoring='f1_macro',
    cv=3,
    random_state=222,
    verbose=2,
    n_jobs=-1
)

rs_DROPOUT_af.fit(X_train_af_scaled, y_train_af, epochs=200,
                  validation_split=0.1, callbacks=[es_af])
best_model_DROPOUT_af = rs_DROPOUT_af.best_estimator_
best_model_DROPOUT_af.model_.save('best_model_DROPOUT_af.keras')


# COMPARAZIONE FINALE MODELLI "AF"
best_NOREG_af_loaded = load_model('best_model_NOREG_af.keras')
best_L1_af_loaded = load_model('best_model_L1_af.keras')
best_L2_af_loaded = load_model('best_model_L2_af.keras')
best_DROPOUT_af_loaded = load_model('best_model_DROPOUT_af.keras')

loss_NOREG_af, acc_NOREG_af = best_NOREG_af_loaded.evaluate(
    X_test_af_scaled, y_test_af, verbose=0)
loss_L1_af, acc_L1_af = best_L1_af_loaded.evaluate(
    X_test_af_scaled, y_test_af, verbose=0)
loss_L2_af, acc_L2_af = best_L2_af_loaded.evaluate(
    X_test_af_scaled, y_test_af, verbose=0)
loss_DROPOUT_af, acc_DROPOUT_af = best_DROPOUT_af_loaded.evaluate(
    X_test_af_scaled, y_test_af, verbose=0)

pred_NOREG_af = np.argmax(best_NOREG_af_loaded.predict(
    X_test_af_scaled, verbose=0), axis=1)
pred_L1_af = np.argmax(best_L1_af_loaded.predict(
    X_test_af_scaled, verbose=0), axis=1)
pred_L2_af = np.argmax(best_L2_af_loaded.predict(
    X_test_af_scaled, verbose=0), axis=1)
pred_DROPOUT_af = np.argmax(best_DROPOUT_af_loaded.predict(
    X_test_af_scaled, verbose=0), axis=1)

f1_NOREG_af = f1_score(y_test_af, pred_NOREG_af, average='macro')
f1_L1_af = f1_score(y_test_af, pred_L1_af, average='macro')
f1_L2_af = f1_score(y_test_af, pred_L2_af, average='macro')
f1_DROPOUT_af = f1_score(y_test_af, pred_DROPOUT_af, average='macro')

print("\n### COMPARAZIONE FINALE MODELLI DNN (AF) ###\n")
print(f"{'Modello':<25} | {'Loss':<8} | {'Accuracy':<10} | {'F1-Macro':<10}")
print("-" * 65)
print(f"{'No Regolarizzazione':<25} | {loss_NOREG_af:<8.4f} | {acc_NOREG_af:<10.4f} | {f1_NOREG_af:<10.4f}")
print(f"{'L1 (Lasso)':<25} | {loss_L1_af:<8.4f} | {acc_L1_af:<10.4f} | {f1_L1_af:<10.4f}")
print(f"{'L2 (Ridge)':<25} | {loss_L2_af:<8.4f} | {acc_L2_af:<10.4f} | {f1_L2_af:<10.4f}")
print(f"{'Dropout':<25} | {loss_DROPOUT_af:<8.4f} | {acc_DROPOUT_af:<10.4f} | {f1_DROPOUT_af:<10.4f}")

c:\Users\Riccardo\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 16 is smaller than n_iter=20. Running 16 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 16 candidates, totalling 48 fits


c:\Users\Riccardo\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits

### COMPARAZIONE FINALE MODELLI DNN (AF) ###

Modello                   | Loss     | Accuracy   | F1-Macro  
-----------------------------------------------------------------
No Regolarizzazione       | 3.5042   | 0.6642     | 0.4984    
L1 (Lasso)                | 0.8780   | 0.7444     | 0.6363    
L2 (Ridge)                | 1.0280   | 0.7201     | 0.5680    
Dropout                   | 1.2957   | 0.7015     | 0.4792    


In [14]:
# Logistica

# Inizializzazione del modello base
logistica_base = LogisticRegression(
    random_state=222,
    class_weight='balanced',
    multi_class='multinomial',
    max_iter=1000,
    solver='lbfgs',  # algoritmo specializzato per l2; prima usavo saga
    penalty='l2',  # facendo delle prove, l2 è meglio di l1
    tol=1e-3
)

# Iperparametri
parametri_log = {
    # regolarizzazione della logistica
    'C': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]
}

# Grid Search
gs_logistica = GridSearchCV(
    estimator=logistica_base,
    param_grid=parametri_log,
    scoring='f1_macro',
    cv=10,
    verbose=1,
    n_jobs=-1
)

# Training
gs_logistica.fit(X_train_scaled, y_train)

# Risultati
best_logistica = gs_logistica.best_estimator_

print("### CONFIGURAZIONE OTTIMALE LOGISTICA ###")
for parametro, valore in gs_logistica.best_params_.items():
    print(f" - {parametro}: {valore}")

print("\n### REPORT DI CLASSIFICAZIONE (TEST SET) ###")
y_pred_log = best_logistica.predict(X_test_scaled)
print(classification_report(y_test, y_pred_log))


##########
# Uguale ma per Ore

logistica_base_ore = LogisticRegression(
    random_state=222,
    class_weight='balanced',
    multi_class='multinomial',
    max_iter=1000,
    solver='lbfgs',
    penalty='l2',
    tol=1e-3
)

gs_logistica_ore = GridSearchCV(
    estimator=logistica_base_ore,
    param_grid=parametri_log,
    scoring='f1_macro',
    cv=10,
    verbose=1,
    n_jobs=-1
)

# Training (Ore)
gs_logistica_ore.fit(X_train_ore_scaled, y_train_ore)

# Risultati (Ore)
best_logistica_ore = gs_logistica_ore.best_estimator_

print("### CONFIGURAZIONE OTTIMALE LOGISTICA (ORE) ###")
for parametro, valore in gs_logistica_ore.best_params_.items():
    print(f" - {parametro}: {valore}")

print("\n### REPORT DI CLASSIFICAZIONE (TEST SET - ORE) ###")
y_pred_log_ore = best_logistica_ore.predict(X_test_ore_scaled)
print(classification_report(y_test_ore, y_pred_log_ore))


##############
# Uguale ma AF

logistica_base_af = LogisticRegression(
    random_state=222,
    class_weight='balanced',
    multi_class='multinomial',
    max_iter=1000,
    solver='lbfgs',
    penalty='l2',
    tol=1e-3
)

gs_logistica_af = GridSearchCV(
    estimator=logistica_base_af,
    param_grid=parametri_log,
    scoring='f1_macro',
    cv=10,
    verbose=1,
    n_jobs=-1
)

# Training (AF)
gs_logistica_af.fit(X_train_af_scaled, y_train_af)

# Risultati (AF)
best_logistica_af = gs_logistica_af.best_estimator_

print("\n### CONFIGURAZIONE OTTIMALE LOGISTICA (AF) ###")
for parametro, valore in gs_logistica_af.best_params_.items():
    print(f" - {parametro}: {valore}")

print("\n### REPORT DI CLASSIFICAZIONE (TEST SET - AF) ###")
y_pred_log_af = best_logistica_af.predict(X_test_af_scaled)
print(classification_report(y_test_af, y_pred_log_af))

Fitting 10 folds for each of 7 candidates, totalling 70 fits


c:\Users\Riccardo\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


### CONFIGURAZIONE OTTIMALE LOGISTICA ###
 - C: 0.001

### REPORT DI CLASSIFICAZIONE (TEST SET) ###
              precision    recall  f1-score   support

         0.0       0.83      0.64      0.72      1742
         1.0       0.28      0.22      0.25       450
         2.0       0.22      0.35      0.27       261
         3.0       0.01      0.31      0.03        16

    accuracy                           0.53      2469
   macro avg       0.34      0.38      0.32      2469
weighted avg       0.66      0.53      0.58      2469

Fitting 10 folds for each of 7 candidates, totalling 70 fits


c:\Users\Riccardo\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


### CONFIGURAZIONE OTTIMALE LOGISTICA (ORE) ###
 - C: 0.01

### REPORT DI CLASSIFICAZIONE (TEST SET - ORE) ###
              precision    recall  f1-score   support

         0.0       0.70      0.63      0.66      1089
         1.0       0.40      0.50      0.44       539
         2.0       0.37      0.24      0.29       616
         3.0       0.27      0.50      0.35       225

    accuracy                           0.49      2469
   macro avg       0.44      0.47      0.44      2469
weighted avg       0.51      0.49      0.49      2469

Fitting 10 folds for each of 7 candidates, totalling 70 fits

### CONFIGURAZIONE OTTIMALE LOGISTICA (AF) ###
 - C: 10

### REPORT DI CLASSIFICAZIONE (TEST SET - AF) ###
              precision    recall  f1-score   support

         1.0       0.18      0.52      0.27       136
         2.0       0.42      0.44      0.43       555
         3.0       0.82      0.53      0.65      1550
         4.0       0.26      0.49      0.34       199
         5.0  

c:\Users\Riccardo\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
